# TB Portals - DA-MoE **mode = `a1`**  *(heavy / stretch)*

A1's ALP comes from YOLO lesion boxes ∩ MedSAM lung, so this notebook first **retrains YOLOv5 on TBX11K** and **computes detection-ALP for all 5,010 images** before training the MoE. NOTE: a1 mode has a SINGLE detection view, so the mixture is trivial — only DANN + critic act here. A1's signal is more useful fused into the `fusion` notebook (pass the det_alp CSV this notebook writes).

**Anchor vs the locked baselines** (`baseline_runs/BASELINE_COMPARISON.md`) — Timika MAE per country:
- A2: Romania 20.11 / **Moldova 30.68** / Kazakhstan 21.35
- A3: Romania 20.26 / **Moldova 26.16** / Kazakhstan 21.90
- A1: Romania 26.84 / **Moldova 32.76** / Kazakhstan 21.87

Moldova is the target.

**Attach datasets:** `tb-portals-cxr-pngs`, `medsam-vit-b`, `tbx-11`.

## 0 - Clone the codebase  *(restart kernel after any pull that changed .py)*

In [ ]:
import os, sys, subprocess
REPO_URL = "https://github.com/mabdullahi7780/dl-project-codebase.git"
REPO_DIR = "/kaggle/working/dl-project-codebase"
BRANCH   = "cleaned-repo"
if os.path.isdir(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, REPO_DIR], check=True)
for _p in (REPO_DIR, REPO_DIR + "/scripts"):
    if _p not in sys.path:
        sys.path.insert(0, _p)
print("repo ready at", REPO_DIR)
# After a git pull that changed .py modules, RESTART the kernel so Python reloads them.

## Install deps

In [ ]:
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "segment-anything", "pydicom", "pylibjpeg", "pylibjpeg-libjpeg", "ultralytics"], check=False)
print("deps installed")

## Paths - edit dataset slugs if yours differ

In [ ]:
import os
WORK           = "/kaggle/working"
REPO_DIR       = "/kaggle/working/dl-project-codebase"
DATASET        = "/kaggle/input/datasets/mabdullahi454/tb-portals-cxr-pngs"
KAGGLE_EXPORT  = f"{DATASET}/kaggle_export"
MEDSAM_CKPT    = "/kaggle/input/datasets/iahmedhabib/medsam-vit-b/medsam_vit_b.pth"
LUNG_DECODER   = f"{REPO_DIR}/checkpoints/component4/component4_mask_decoder.pt"
PAPER_MANIFEST = f"{WORK}/tbportals_manifest_paper.csv"
CROPS_DIR      = f"{WORK}/crops"
TBX_ROOT  = "/kaggle/input/datasets/usmanshams/tbx-11/TBX11K"
YOLO_DIR  = f"{WORK}/tbx11k_yolo"
YOLO_BEST = f"{WORK}/yolo_runs/tbx11k/weights/best.pt"
print("KAGGLE_EXPORT:", KAGGLE_EXPORT, "->", os.path.isdir(KAGGLE_EXPORT))
print("MEDSAM_CKPT:  ", MEDSAM_CKPT, "->", os.path.isfile(MEDSAM_CKPT))
print("LUNG_DECODER: ", LUNG_DECODER, "->", os.path.isfile(LUNG_DECODER))

## 1 - Build the 5,010-image manifest (Kantipudi Table 1)

In [ ]:
import sys, pandas as pd
from pathlib import Path
if REPO_DIR + "/scripts" not in sys.path: sys.path.insert(0, REPO_DIR + "/scripts")
from build_paper_manifest import subsample, PAPER_TOTAL
raw = pd.read_csv(f"{KAGGLE_EXPORT}/manifest.csv",
                  dtype={"image_id": str, "patient_id": str, "country": str})
raw["image_path"] = raw["image_path"].apply(
    lambda p: p if str(p).startswith("/") else f"{KAGGLE_EXPORT}/{p}")
paper_df = subsample(raw, seed=42)
paper_df["image_id"] = paper_df["image_path"].apply(lambda p: Path(str(p)).stem)
paper_df.to_csv(PAPER_MANIFEST, index=False)
print(f"Paper manifest: {len(paper_df)} images (target {PAPER_TOTAL}) -> {PAPER_MANIFEST}")

## 2 - MedSAM lung crops (~25 min first time; idempotent)

In [ ]:
import os, sys
if REPO_DIR + "/scripts" not in sys.path: sys.path.insert(0, REPO_DIR + "/scripts")
from cache_lung_crops import main as crops_main
argv = ["--manifest", PAPER_MANIFEST, "--out-dir", CROPS_DIR,
        "--medsam-ckpt", MEDSAM_CKPT, "--size", "224", "--pad", "32"]
if os.path.isfile(LUNG_DECODER): argv += ["--lung-decoder-ckpt", LUNG_DECODER]
crops_main(argv)
print("crops ->", CROPS_DIR, "| count:", len(os.listdir(CROPS_DIR)))

## 3 - Convert TBX11K -> YOLO (paper's official split)

In [ ]:
import sys
if REPO_DIR + "/scripts" not in sys.path: sys.path.insert(0, REPO_DIR + "/scripts")
from prepare_tbx11k_yolo import main as prep_main
prep_main(["--tbx-root", TBX_ROOT, "--out", YOLO_DIR,
           "--train-list", f"{TBX_ROOT}/lists/TBX11K_train.txt",
           "--val-list",   f"{TBX_ROOT}/lists/TBX11K_val.txt"])

## 4 - Train YOLOv5 lesion detector (~30-60 min)

In [ ]:
from ultralytics import YOLO
yolo = YOLO("yolov5nu.pt")
yolo.train(data=f"{YOLO_DIR}/tbx11k.yaml", epochs=100, imgsz=640, batch=16,
           project=f"{WORK}/yolo_runs", name="tbx11k", exist_ok=True)
print("best weights ->", YOLO_BEST)

## 5 - Compute detection-ALP over all images (~30-40 min)

In [ ]:
# Detection-ALP per image = |YOLO lesion boxes ∩ MedSAM lung| / |lung| * 100.
# Computed once over the whole manifest so the MoE a1 view can look it up by image_id.
import os, pandas as pd, torch
from ultralytics import YOLO
from src.components.component4_lung import Component4MedSAM
from src.training.train_a1_detect import detection_alp
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
yolo = YOLO(YOLO_BEST)
medsam = Component4MedSAM(backend="medsam", checkpoint_path=MEDSAM_CKPT).to(device)
if os.path.isfile(LUNG_DECODER): medsam.load_trained_decoder(LUNG_DECODER)
medsam.eval()
df = pd.read_csv(PAPER_MANIFEST, dtype={"image_id": str})
rows = []
for i, r in df.iterrows():
    rows.append({"image_id": r["image_id"], "det_alp": detection_alp(r["image_path"], medsam, yolo, device, 0.25)})
    if i % 500 == 0: print(f"  det_alp {i}/{len(df)}")
DET_ALP_CSV = f"{WORK}/det_alp.csv"
pd.DataFrame(rows).to_csv(DET_ALP_CSV, index=False)
del medsam, yolo
torch.cuda.empty_cache()
print("det_alp ->", DET_ALP_CSV, "| n =", len(rows))

## 6 - Configure

In [ ]:
# ---- this notebook is dedicated to MODE = "a1" --------------------------
MODE     = "a1"
SEEDS    = ["0", "1", "2"]   # full run; use ["0"] first if you want a fast sanity check
EPOCHS   = "30"
PRETRAIN = "10"              # phase-1 expert-pretraining epochs (rest = gate+critic+DANN)
OUT_DIR  = f"/kaggle/working/checkpoints/moe_{MODE}"
import os; os.makedirs(OUT_DIR, exist_ok=True)
print("will run MoE mode =", MODE, "seeds =", SEEDS, "->", OUT_DIR)

## 7 - Train + evaluate the MoE (a1 view + cavity agent + DANN + critic)

In [ ]:
from src.training.train_da_moe import main as moe_main
argv = ["--mode", MODE, "--manifest", PAPER_MANIFEST, "--crops-dir", CROPS_DIR,
        "--out-dir", OUT_DIR, "--held-outs", "Romania", "Moldova", "Kazakhstan",
        "--seeds", *SEEDS, "--epochs", EPOCHS, "--pretrain-epochs", PRETRAIN,
        "--det-alp-csv", DET_ALP_CSV,
        "--batch-size", "60", "--accum-steps", "5", "--num-workers", "2"]
# a1/a2/fusion use the cavity agent on whole images (matches the locked A2 config):
if MODE in ("a1", "a2", "fusion"):
    argv += ["--cavity-no-lung-crop"]
moe_main(argv)

## 8 - Save results + trained models

In [ ]:
import os, shutil
# results CSV alone (small — quick to grab)
csv_dst = f"/kaggle/working/results_moe_{MODE}.csv"
shutil.copy(f"{OUT_DIR}/results_moe.csv", csv_dst)
# trained models + results -> one zip (the moe_*.pt checkpoints live in OUT_DIR)
zip_path = shutil.make_archive(f"/kaggle/working/checkpoints_moe_{MODE}", "zip", OUT_DIR)
n_ckpt = len([f for f in os.listdir(OUT_DIR) if f.endswith(".pt")])
print("Saved:")
print("  ", csv_dst, " (results only)")
print("  ", zip_path, f" ({n_ckpt} trained .pt models + results_moe.csv)")
print("Download BOTH from the Output panel. Drop the CSV into baseline_runs/MoE/.")

## 9 - Ablations (optional, run last)

In [ ]:
# ---- ablations (run last, after the full model beats the baseline) ----------
from src.training.train_da_moe import main as moe_main
def run(tag, extra):
    import os
    out = f"/kaggle/working/checkpoints/moe_{MODE}_abl_{tag}"
    os.makedirs(out, exist_ok=True)
    base = ["--mode", MODE, "--manifest", PAPER_MANIFEST, "--crops-dir", CROPS_DIR,
            "--out-dir", out, "--held-outs", "Romania", "Moldova", "Kazakhstan",
            "--seeds", "0", "--epochs", EPOCHS, "--pretrain-epochs", PRETRAIN,
            "--batch-size", "60", "--accum-steps", "5", "--num-workers", "2"]
    if MODE in ("a1", "a2", "fusion"): base += ["--cavity-no-lung-crop"]
    if "DET_ALP_CSV" in globals(): base += ["--det-alp-csv", DET_ALP_CSV]
    print("\n==== ABLATION", MODE, tag, extra, "===="); moe_main(base + extra)

run("no_dann",   ["--no-dann"])
run("no_critic", ["--no-critic"])
run("no_both",   ["--no-dann", "--no-critic"])